# Notebook 01 (Participant): Train + Generate with EngiOpt CGAN-2D


In [ ]:
# Colab/local dependency bootstrap
import subprocess
import sys

IN_COLAB = 'google.colab' in sys.modules
FORCE_INSTALL = False  # Set True to force reinstall outside Colab
PACKAGES = ['engibench[beams2d]', 'sqlitedict', 'torch', 'torchvision', 'matplotlib', 'pandas', 'tqdm', 'tyro', 'wandb']
ENGIOPT_GIT = 'git+https://github.com/IDEALLab/EngiOpt.git@codex/dcc26-workshop-notebooks#egg=engiopt'

if IN_COLAB or FORCE_INSTALL:
    print('Installing base dependencies...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *PACKAGES])
    print('Installing EngiOpt from GitHub branch...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', ENGIOPT_GIT])
    print('Dependency install complete.')
else:
    print('Skipping install (using current environment).')


In [ ]:
import json
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch as th
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from engibench.problems.beams2d.v0 import Beams2D

try:
    from engiopt.cgan_2d.cgan_2d import Generator as EngiOptCGAN2DGenerator
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        'Could not import engiopt model class. Run the bootstrap cell first; on Colab, restart runtime after install if needed.'
    ) from exc


def resolve_artifact_dir(create: bool = False) -> Path:
    in_colab = 'google.colab' in sys.modules

    if in_colab:
        try:
            from google.colab import drive

            if not Path('/content/drive/MyDrive').exists():
                print('Mounting Google Drive for persistent workshop artifacts...')
                drive.mount('/content/drive', force_remount=False)
        except Exception as exc:
            print('Drive mount skipped/failed, falling back to runtime storage:', exc)

        drive_artifacts = Path('/content/drive/MyDrive/dcc26_workshop/artifacts')
        runtime_artifacts = Path('/content/workshops/dcc26/artifacts')

        target = drive_artifacts if Path('/content/drive/MyDrive').exists() else runtime_artifacts
        if create:
            target.mkdir(parents=True, exist_ok=True)
        return target

    local_artifacts = Path('workshops/dcc26/artifacts')
    if create:
        local_artifacts.mkdir(parents=True, exist_ok=True)
    return local_artifacts


SEED = 7
random.seed(SEED)
np.random.seed(SEED)
th.manual_seed(SEED)
if th.cuda.is_available():
    th.cuda.manual_seed_all(SEED)

DEVICE = th.device('cuda' if th.cuda.is_available() else 'cpu')
print('device:', DEVICE)

ARTIFACT_DIR = resolve_artifact_dir(create=True)
print('artifact dir:', ARTIFACT_DIR)

CKPT_PATH = ARTIFACT_DIR / 'engiopt_cgan2d_generator_supervised.pt'
LATENT_DIM = 32


In [ ]:
problem = Beams2D(seed=SEED)
train_ds = problem.dataset['train']
test_ds = problem.dataset['test']

condition_keys = problem.conditions_keys
print('condition keys:', condition_keys)

# Build compact train subset to keep runtime stable in workshop
N_TRAIN = 512
subset_idx = np.random.default_rng(SEED).choice(len(train_ds), size=N_TRAIN, replace=False)

conds_np = np.stack([np.array(train_ds[k])[subset_idx].astype(np.float32) for k in condition_keys], axis=1)
designs_np = np.array(train_ds['optimal_design'])[subset_idx].astype(np.float32)

# EngiOpt CGAN generator emits tanh-scaled outputs in [-1, 1]
targets_np = (designs_np * 2.0) - 1.0

print('conditions shape:', conds_np.shape)
print('designs shape:', designs_np.shape)
print('target range:', float(targets_np.min()), 'to', float(targets_np.max()))


In [ ]:
# TODO 1: instantiate EngiOpt CGAN-2D generator + optimizer/loss
# Required class is already imported as EngiOptCGAN2DGenerator
#
# Suggested:
# model = EngiOptCGAN2DGenerator(latent_dim=LATENT_DIM, n_conds=conds_np.shape[1], design_shape=problem.design_space.shape).to(DEVICE)
# optimizer = th.optim.Adam(model.parameters(), lr=1e-3)
# criterion = nn.MSELoss()
#
# def sample_noise(batch_size: int) -> th.Tensor:
#     return th.randn((batch_size, LATENT_DIM), device=DEVICE, dtype=th.float32)

raise NotImplementedError('Complete TODO 1 with the EngiOpt model setup')


In [ ]:
TRAIN_FROM_SCRATCH = True
EPOCHS = 8
BATCH_SIZE = 64

if TRAIN_FROM_SCRATCH:
    # TODO 2: implement lightweight supervised training loop for EngiOpt generator
    # - dataset tensors: conds_np -> input conditions, targets_np -> tanh-scaled targets in [-1,1]
    # - sample latent noise each batch with sample_noise(...)
    # - loss = criterion(pred, target_batch)
    # - optimizer step
    # - save checkpoint dict with keys: model, condition_keys, latent_dim, model_family
    raise NotImplementedError('Complete TODO 2 training loop')
elif CKPT_PATH.exists():
    ckpt = th.load(CKPT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt['model'])
    model.eval()
    print('loaded checkpoint from', CKPT_PATH)
else:
    print('No checkpoint found. Use fallback generation cell below.')


In [ ]:
# TODO 3: implement generation path with EngiOpt generator
# - sample N_SAMPLES test conditions and baseline designs
# - if USE_NEAREST_NEIGHBOR_FALLBACK: nearest-neighbor designs from training subset
# - else: run model(sample_noise(...), condition_tensor)
#   and map tanh outputs to design space with ((x + 1)/2).clamp(0, 1)
# - set gen_designs, baseline_designs, test_conds

USE_NEAREST_NEIGHBOR_FALLBACK = False

raise NotImplementedError('Complete TODO 3 generation path')


In [ ]:
# TODO 5: serialize artifacts for Notebook 02
# - generated_designs.npy
# - baseline_designs.npy
# - conditions.json

raise NotImplementedError('Complete TODO 5 artifact export')

In [ ]:
# Quick visual check of generated designs
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(gen_designs[i], cmap='gray', vmin=0, vmax=1)
    ax.axis('off')
    ax.set_title(f'gen {i}')
plt.tight_layout()
plt.show()

## Next

Continue with **Notebook 02** to validate and evaluate generated designs against baselines.
